# 02 — Pilot SAE Training on DINOv2 Activations

Trains a BatchTopK SAE on the cached DINOv2 ViT-B/14 layer-11 activations as a sanity check.

**Pilot config:** 10K images × 256 patches = 2.56M patch tokens, 16× expansion (dict_size=12288), k=192.

In [6]:
# ── Cell 1: Load stats and discover shards ────────────────────────────────────
import glob
import json
import os
import sys

import torch

# Make sure the repo root is on the path when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils.paths import ACTIVATION_ROOT

ACTIVATION_DIR = os.path.join(ACTIVATION_ROOT, "dinov2_vitb14", "layer_11")

# Load stats
stats_path = os.path.join(ACTIVATION_DIR, "stats.json")
with open(stats_path) as f:
    stats = json.load(f)

mean = torch.tensor(stats["mean"], dtype=torch.float32)  # [768]
std  = float(stats["std"])                                # scalar
print(f"Stats loaded — mean shape: {mean.shape},  std: {std:.4f}")
print(f"Backbone: {stats.get('backbone', 'dinov2_vitb14')}  "
      f"layer: {stats.get('layer', 11)}  "
      f"num_images: {stats.get('num_images')}  "
      f"d_model: {stats.get('d_model')}")

# Discover shards — do NOT load them yet
shard_paths = sorted(glob.glob(os.path.join(ACTIVATION_DIR, "shard_*.pt")))
print(f"\nFound {len(shard_paths)} shard(s)")
print("Shards will be loaded one at a time during training to stay within RAM limits.")

Stats loaded — mean shape: torch.Size([768]),  std: 1.8910
Backbone: dinov2_vitb14  layer: 11  num_images: 100000  d_model: 768

Found 20 shard(s)
Shards will be loaded one at a time during training to stay within RAM limits.


In [7]:
# ── Cell 2: Configure and instantiate BatchTopK SAE ───────────────────────────
from overcomplete import BatchTopKSAE, TopKSAE
from src.training.overcomplete_config import make_sae_config

D_MODEL          = 768
EXPANSION_FACTOR = 16
DICT_SIZE        = D_MODEL * EXPANSION_FACTOR  # 12288
K                = 192   # active features per batch step
THRESHOLD_MOM    = 0.9
BATCH_SIZE = 4096

device = "cuda" if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

config = make_sae_config(
    d_model=D_MODEL,
    expansion_factor=EXPANSION_FACTOR,
    k=K*BATCH_SIZE,
    architecture="batchtopk",
)
print(f"Config: {config}")

# Instantiate via overcomplete and call .tied() to use dictionary^T as the encoder.
# The default MLPEncoder uses BatchNorm1d(12288) which is very slow on CPU/MPS.
# Tied mode is standard SAE practice and avoids this overhead entirely.
sae = BatchTopKSAE(
    input_shape=D_MODEL,
    nb_concepts=DICT_SIZE,
    top_k=K*BATCH_SIZE,
    device=device
)

num_params = sum(p.numel() for p in sae.parameters())
print(f"\nBatchTopKSAE instantiated:")
print(f"  input_shape:  {D_MODEL}")
print(f"  nb_concepts:  {DICT_SIZE}  ({EXPANSION_FACTOR}× expansion)")
print(f"  top_k:        {K}")
print(f"  parameters:   {num_params:,}")

Device: cuda
Config: {'architecture': 'batchtopk', 'd_model': 768, 'expansion_factor': 16, 'dict_size': 12288, 'k': 786432, 'constructor_kwargs': {'input_shape': 768, 'nb_concepts': 12288, 'top_k': 786432, 'threshold_momentum': 0.9}}

BatchTopKSAE instantiated:
  input_shape:  768
  nb_concepts:  12288  (16× expansion)
  top_k:        192
  parameters:   18,886,656


In [ ]:
# ── Cell 1b: Quick diagnostic — single forward pass before training ────────────
# Load one shard temporarily to verify the SAE initialisation looks reasonable.
_diag_shard = torch.load(shard_paths[0], map_location="cpu", weights_only=True)
_diag_tokens = (_diag_shard[:1].reshape(-1, _diag_shard.shape[-1]).float() - mean) / std
# (we only need a tiny slice — just 256 tokens is enough)
_diag_tokens = _diag_tokens[:256]

# Need the SAE to exist — this cell must run after Cell 2
try:
    sae.eval()
    with torch.no_grad():
        test_batch = _diag_tokens.to(device)
        pre_codes, codes, x_hat = sae(test_batch)
        print(f"pre_codes — min: {pre_codes.min():.4f}, max: {pre_codes.max():.4f}, mean: {pre_codes.mean():.4f}")
        print(f"codes nonzero: {(codes != 0).sum()}/{codes.numel()}")
        print(f"x_hat — min: {x_hat.min():.4f}, max: {x_hat.max():.4f}, std: {x_hat.std():.4f}")
        print(f"input — min: {test_batch.min():.4f}, max: {test_batch.max():.4f}, std: {test_batch.std():.4f}")
    del test_batch, pre_codes, codes, x_hat
except NameError:
    print("Skip: run Cell 2 first to instantiate `sae`.")
del _diag_shard, _diag_tokens

In [8]:
# ── Cell 3: Train the SAE (shard-by-shard to stay within RAM) ─────────────────
import random
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

BATCH_SIZE  = BATCH_SIZE
LR          = 1e-3
NUM_EPOCHS  = 4
LOG_EVERY   = 50

optimizer   = torch.optim.Adam(sae.parameters(), lr=LR)
training_log = []
global_step  = 0

# Estimate steps for progress reporting
approx_tokens_per_shard = 5000 * 256  # shard_size * patches
steps_per_shard = (approx_tokens_per_shard + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = NUM_EPOCHS * len(shard_paths) * steps_per_shard
print(f"Training: {NUM_EPOCHS} epochs × {len(shard_paths)} shards  (~{total_steps} steps estimated)")
print(f"Batch size: {BATCH_SIZE}  |  LR: {LR}")
print("Each shard is loaded, trained on, then freed — max RAM use ~3 GB per shard.\n")

sae.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    epoch_steps = 0

    # Shuffle shard order each epoch for better coverage
    epoch_shards = shard_paths[:]
    random.shuffle(epoch_shards)

    for shard_path in tqdm(epoch_shards, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} shards", leave=False):
        # Load shard, normalize, flatten, shuffle
        shard = torch.load(shard_path, map_location="cpu", weights_only=True)  # [N, 256, 768]
        N, P, D = shard.shape
        tokens = shard.reshape(N * P, D).float()
        tokens = (tokens - mean) / std
        perm = torch.randperm(tokens.shape[0])
        tokens = tokens[perm]

        loader = DataLoader(TensorDataset(tokens), batch_size=BATCH_SIZE,
                            shuffle=False, drop_last=False, num_workers=0)
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            _pre_codes, codes, x_hat = sae(batch)
            loss = (batch - x_hat).square().mean()
            loss.backward()
            optimizer.step()

            loss_val     = float(loss.item())
            epoch_loss  += loss_val
            epoch_steps += 1
            global_step += 1

            if global_step % LOG_EVERY == 0:
                training_log.append({"step": global_step, "loss": loss_val})

        del shard, tokens, loader

    avg = epoch_loss / epoch_steps if epoch_steps > 0 else float("nan")
    print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  |  step {global_step:>6}  |  avg loss: {avg:.4f}")

print("\nTraining complete.")

/mnt/NAS/home/ds5725/visaebench-internal/.visaebench/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training: 4 epochs × 20 shards  (~25040 steps estimated)
Batch size: 4096  |  LR: 0.001
Each shard is loaded, trained on, then freed — max RAM use ~3 GB per shard.



Epoch 1/4 shards:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch  1/4  |  step   6260  |  avg loss: 0.1184


Epoch  2/4  |  step  12520  |  avg loss: 0.0920


Epoch  3/4  |  step  18780  |  avg loss: 0.0885


Epoch  4/4  |  step  25040  |  avg loss: 0.0872

Training complete.


In [ ]:
# ── Cell 4: Save checkpoint (before eval — so weights are safe if eval OOMs) ───
import yaml

from src.utils.paths import CHECKPOINT_ROOT

CHECKPOINT_DIR = os.path.join(CHECKPOINT_ROOT, "dinov2_vitb14", "batchtopk_16x_k192")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

sae_path = os.path.join(CHECKPOINT_DIR, "sae.pt")
# Include running_threshold for BatchTopKSAE — it's a plain attribute,
# not a registered buffer, so state_dict() doesn't capture it.
save_dict = sae.state_dict()
if hasattr(sae, "running_threshold") and sae.running_threshold is not None:
    save_dict["_running_threshold"] = sae.running_threshold.detach().cpu()
torch.save(save_dict, sae_path)
print(f"Saved SAE weights  → {sae_path}")

cfg = {
    "backbone":         "dinov2_vitb14",
    "architecture":     "batchtopk",
    "d_model":          D_MODEL,
    "expansion_factor": EXPANSION_FACTOR,
    "dict_size":        DICT_SIZE,
    "k":                K,
    "lr":               LR,
    "batch_size":       BATCH_SIZE,
    "num_epochs":       NUM_EPOCHS,
    "total_steps":      global_step,
    "activation_dir":   ACTIVATION_DIR,
}
config_path = os.path.join(CHECKPOINT_DIR, "config.yaml")
with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Saved config       → {config_path}")

log_path = os.path.join(CHECKPOINT_DIR, "training_log.json")
with open(log_path, "w") as f:
    json.dump({"training_loss": training_log}, f, indent=2)
print(f"Saved training log → {log_path}")
print("\nCheckpoint saved. Run Cell 5 to evaluate.")

In [11]:
# ── Cell 5: Evaluate basic metrics (fully incremental — no tensor accumulation) ─
from tqdm.auto import tqdm

EVAL_BATCH   = 512   # small batches to keep peak RAM low
EVAL_SHARDS  = shard_paths[-4:]  # last 4 shards as held-out eval set

# temp code #################################
val_shard_path = '/mnt/NAS/data/ds5725/visaebench/activations/dinov2_vitb14/layer_11_val'
EVAL_SHARDS = shard_paths = sorted(glob.glob(os.path.join(val_shard_path, "shard_*.pt")))
# temp code #################################


sae.eval()
print(f"Evaluating on {len(EVAL_SHARDS)} held-out shards (~{len(EVAL_SHARDS)*5000} images)...")

# ── Single pass: FVU, L0, and dead features — all incremental ──
n_tok    = 0
sum_x2   = 0.0
sum_x    = 0.0
sum_res2 = 0.0
sum_res  = 0.0
sum_l0   = 0.0
ever_active = torch.zeros(DICT_SIZE, dtype=torch.bool)

with torch.no_grad():
    for shard_path in tqdm(EVAL_SHARDS, desc="FVU/L0/dead eval"):
        shard = torch.load(shard_path, map_location="cpu", weights_only=True)
        N, P, D = shard.shape
        tokens = shard.reshape(N * P, D).float()
        tokens = (tokens - mean) / std
        for start in range(0, len(tokens), EVAL_BATCH):
            batch = tokens[start : start + EVAL_BATCH].to(device)
            _pre, codes, x_hat = sae(batch)
            res = batch - x_hat
            sum_x2   += float(batch.pow(2).sum())
            sum_x    += float(batch.sum())
            sum_res2 += float(res.pow(2).sum())
            sum_res  += float(res.sum())
            sum_l0   += float((codes != 0).float().sum(dim=1).sum())
            ever_active |= (codes != 0).any(dim=0).cpu()
            n_tok    += batch.shape[0]
            del batch, _pre, codes, x_hat, res
        del shard, tokens

if device == "cuda":
    torch.cuda.empty_cache()

var_x   = sum_x2 / n_tok - (sum_x / n_tok) ** 2
var_res = sum_res2 / n_tok - (sum_res / n_tok) ** 2
fvu = float(var_res / var_x)
l0  = float(sum_l0 / n_tok)
dead_count = int((~ever_active).sum())
dead_pct   = 100.0 * dead_count / DICT_SIZE

print("=" * 50)
print("Evaluation Metrics")
print("=" * 50)
print(f"  FVU (Fraction of Variance Unexplained): {fvu:.4f}   (target < 0.10)")
print(f"  L0  (avg active features / input):      {l0:.1f}    (target ≈ {K})")
print(f"  Dead features: {dead_count} / {DICT_SIZE}  ({dead_pct:.1f}%)  (target < 10%)")
print("=" * 50)

# Append final metrics to the already-saved training log
with open(log_path) as f:
    saved_log = json.load(f)
saved_log["final_metrics"] = {
    "fvu": fvu, "l0": l0, "dead_features": dead_count, "dead_pct": dead_pct
}
with open(log_path, "w") as f:
    json.dump(saved_log, f, indent=2)
print(f"Final metrics appended → {log_path}")

Evaluating on 2 held-out shards (~10000 images)...


FVU/L0/dead eval: 100%|██████████| 2/2 [00:47<00:00, 23.97s/it]

Evaluation Metrics
  FVU (Fraction of Variance Unexplained): 0.1009   (target < 0.10)
  L0  (avg active features / input):      208.0    (target ≈ 192)
  Dead features: 172 / 12288  (1.4%)  (target < 10%)
Final metrics appended → /mnt/NAS/data/ds5725/visaebench/checkpoints/dinov2_vitb14/batchtopk_16x_k192/training_log.json


In [ ]:
# ── Cell 6 (optional): Load saved checkpoint to resume eval without retraining ─
# Run this cell instead of Cell 3+4 if the kernel crashed after training was done.
import yaml

from src.utils.paths import CHECKPOINT_ROOT

CHECKPOINT_DIR = os.path.join(CHECKPOINT_ROOT, "dinov2_vitb14", "batchtopk_16x_k192")
sae_path       = os.path.join(CHECKPOINT_DIR, "sae.pt")
log_path       = os.path.join(CHECKPOINT_DIR, "training_log.json")

state_dict = torch.load(sae_path, map_location=device, weights_only=True)

# Restore running_threshold for BatchTopKSAE (saved under special key)
saved_threshold = state_dict.pop("_running_threshold", None)
sae.load_state_dict(state_dict)
if saved_threshold is not None:
    sae.running_threshold = saved_threshold.to(device)
else:
    # Old checkpoint — calibrate with a dummy forward pass
    sae.train()
    with torch.no_grad():
        sae(torch.randn(BATCH_SIZE, D_MODEL, device=device))
sae.eval()
print(f"Loaded checkpoint from {sae_path}")

with open(os.path.join(CHECKPOINT_DIR, "config.yaml")) as f:
    saved_cfg = yaml.safe_load(f)

global_step  = saved_cfg.get("total_steps", 0)
training_log = []
print(f"Resumed: global_step={global_step},  device={device}")
print("Now run Cell 5 to evaluate.")